Implementating hybrid search

In [1]:
import pdfplumber

In [2]:
all_text=" "
with pdfplumber.open("sample.pdf") as pdf:
    for page in pdf.pages:
        all_text+=page.extract_text() +"\n"
   

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\admin\Desktop\project\rag-implementation\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=60,add_start_index=True)

In [5]:
chunks=text_splitter.split_text(all_text)

In [6]:
print(chunks[79])

performed on a per-thread basis.
Threads have a similar life cycle to the processes and are mainly managed in the same way. Initially each process is
created with a single thread. However, threads are usually allowed to create new ones using particular system calls.
Then, a thread tree is typically created for each process.
Process
Thread tree.


In [7]:
from sentence_transformers import SentenceTransformer
from sentence_transformers import SparseEncoder
import torch

from qdrant_client import QdrantClient

from qdrant_client.http import models as rest

from dotenv import load_dotenv

import os

load_dotenv()

dense_model=SentenceTransformer("all-MiniLM-L6-v2")
sparse_model=SparseEncoder("prithivida/Splade_PP_en_v2")

client = QdrantClient(
    url=os.getenv("Qdrant_url"),
    prefer_grpc=False 
    
)

client.recreate_collection(
    collection_name=os.getenv("collection_name"),
    vectors_config={
      "dense":rest.VectorParams(size=384, distance=rest.Distance.COSINE)
      },
    sparse_vectors_config={
      "splade":rest.SparseVectorParams()
    }
)

for index,chunk in enumerate  (chunks):
  dense_embedding=dense_model.encode(chunk)
  sparse_embedding=sparse_model.encode(chunk)
  sparse_embedding = sparse_embedding.coalesce()
  indices = sparse_embedding.indices()[0].tolist()
  values = sparse_embedding.values().tolist()
  client.upsert(
     collection_name=os.getenv("collection_name"),
     points=[
        rest.PointStruct(
         id=index,
         vector={
           "dense":dense_embedding.tolist(),
                 "splade":rest.SparseVector(indices=indices,
                                               values=values)
                 },
         payload={"text":chunk}

       )
     ] 
   )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 369.30it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 204/204 [00:00<00:00, 214.31it/s, Materializing param=cls.predictions.transform.dense.weight]                 
The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.prediction

In [8]:
Query="What is a computational model?"
dense_query_embedding=(dense_model.encode(Query)).tolist()

Sparse_query_embedding=(sparse_model.encode(Query)).coalesce()
indices = Sparse_query_embedding.indices()[0].tolist()
values = Sparse_query_embedding.values().tolist()
sparse_query=rest.SparseVector(indices=indices,
                                     values=values)

In [9]:
dense_result=client.query_points(
    collection_name=os.getenv("collection_name"),
    query=dense_query_embedding,
    limit=7,
    with_payload=True,
    using="dense"
)

In [10]:
sparse_result=client.query_points(
    collection_name=os.getenv("collection_name"),
    query=sparse_query,
    limit=7,
    with_payload=True,
    using="splade"
)

In [11]:
print(dense_result)

points=[ScoredPoint(id=0, version=1, score=0.7930154, payload={'text': 'UNIT 1: Computational Models\nThe Concept of Computational Model :- The computer architecture and language classes must have a\ncommon foundation or paradigm called a Computational Model. The concept of a computational model represents\na higher level of abstraction than either the computer architecture or the programming language alone, and covers\nboth, as show below -\nComputational Model\nLevel of\nAbstraction\nComputer Computer\nArchitecture Language'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=1, version=2, score=0.73031145, payload={'text': 'Abstraction\nComputer Computer\nArchitecture Language\nInterpretation of the computational model concept as a high-level abstraction.\nThe Concept of Computation Model - The concept of computational model comprises the set of the following\nthree abstractions –\n1. The basic items of computations\n2. The problem description model\n3. The execution mo

In [12]:
print(sparse_result)

points=[ScoredPoint(id=0, version=1, score=21.403357, payload={'text': 'UNIT 1: Computational Models\nThe Concept of Computational Model :- The computer architecture and language classes must have a\ncommon foundation or paradigm called a Computational Model. The concept of a computational model represents\na higher level of abstraction than either the computer architecture or the programming language alone, and covers\nboth, as show below -\nComputational Model\nLevel of\nAbstraction\nComputer Computer\nArchitecture Language'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=1, version=2, score=20.958376, payload={'text': 'Abstraction\nComputer Computer\nArchitecture Language\nInterpretation of the computational model concept as a high-level abstraction.\nThe Concept of Computation Model - The concept of computational model comprises the set of the following\nthree abstractions –\n1. The basic items of computations\n2. The problem description model\n3. The execution mod

In [ ]:
Rank_dense=[]
dense_points=dense_result.points
for point in dense_points:
    id=point.id
    Rank_dense.append(id)
print(Rank_dense)


[0, 1, 2, 4, 8, 25, 3]


In [ ]:
Rank_sparse=[]
sparse_points=sparse_result.points
for point in sparse_points:
    id=point.id
    
    Rank_sparse.append(id)
print(Rank_sparse)


[0, 1, 2, 4, 8, 25, 3]


In [26]:
def rrf_from_rank(lists_of_id,k=60):
    rff_scores={}
    for lst in lists_of_id:
        for rank,id in enumerate (lst):
            score=1/(k+rank+1)
            if id in rff_scores:
                rff_scores[id]+=score
            else:
                rff_scores[id]=score
        top_5_ascending = sorted(rff_scores.items(), key=lambda x: x[1],reverse=True)[:5]
        return top_5_ascending


In [29]:
top_5=rrf_from_rank([Rank_dense,Rank_sparse])
print(top_5)

[(0, 0.01639344262295082), (1, 0.016129032258064516), (2, 0.015873015873015872), (4, 0.015625), (8, 0.015384615384615385)]


In [ ]:
id_to_text = {}

for p in dense_result.points + sparse_result.points:
    id_to_text[p.id] = p.payload["text"]



{0: 'UNIT 1: Computational Models\nThe Concept of Computational Model :- The computer architecture and language classes must have a\ncommon foundation or paradigm called a Computational Model. The concept of a computational model represents\na higher level of abstraction than either the computer architecture or the programming language alone, and covers\nboth, as show below -\nComputational Model\nLevel of\nAbstraction\nComputer Computer\nArchitecture Language', 1: 'Abstraction\nComputer Computer\nArchitecture Language\nInterpretation of the computational model concept as a high-level abstraction.\nThe Concept of Computation Model - The concept of computational model comprises the set of the following\nthree abstractions –\n1. The basic items of computations\n2. The problem description model\n3. The execution model\nContrary to initial thoughts, the set of abstractions that should be chosen to specify computational models is far from', 4: 'programming languages and are implemented by m

In [ ]:
top_5_text = []
for (lists,score) in top_5:
    text = id_to_text[lists]
    top_5_text.append(text)
print(top_5_text)  


['UNIT 1: Computational Models\nThe Concept of Computational Model :- The computer architecture and language classes must have a\ncommon foundation or paradigm called a Computational Model. The concept of a computational model represents\na higher level of abstraction than either the computer architecture or the programming language alone, and covers\nboth, as show below -\nComputational Model\nLevel of\nAbstraction\nComputer Computer\nArchitecture Language', 'Abstraction\nComputer Computer\nArchitecture Language\nInterpretation of the computational model concept as a high-level abstraction.\nThe Concept of Computation Model - The concept of computational model comprises the set of the following\nthree abstractions –\n1. The basic items of computations\n2. The problem description model\n3. The execution model\nContrary to initial thoughts, the set of abstractions that should be chosen to specify computational models is far from', 'obvious. A smaller number of criteria would define fewe

In [40]:
context="\n\n".join(top_5_text)
print(context)

UNIT 1: Computational Models
The Concept of Computational Model :- The computer architecture and language classes must have a
common foundation or paradigm called a Computational Model. The concept of a computational model represents
a higher level of abstraction than either the computer architecture or the programming language alone, and covers
both, as show below -
Computational Model
Level of
Abstraction
Computer Computer
Architecture Language

Abstraction
Computer Computer
Architecture Language
Interpretation of the computational model concept as a high-level abstraction.
The Concept of Computation Model - The concept of computational model comprises the set of the following
three abstractions –
1. The basic items of computations
2. The problem description model
3. The execution model
Contrary to initial thoughts, the set of abstractions that should be chosen to specify computational models is far from

obvious. A smaller number of criteria would define fewer but more basic computa